- Driver: Controls execution and builds the plan
- Executors: Execute tasks on data partitions
- DAG: Logical execution plan of transformations
DAG (Directed Acyclic Graph) is just:  A step-by-step plan Spark builds before running anything

In [0]:
df_n = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",header=True, inferSchema=True)


in above code What Spark does :
- Reads metadata
- Infers schema
- Does NOT read all data yet

In [0]:
df_n.printSchema()


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [0]:
df_n.count()

67501979

In [0]:
df_n.show(3)

+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code| brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|2019-11-01 00:00:00|      view|   1003461|2053013555631882655|electronics.smart...|xiaomi|489.07|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:00|      view|   5000088|2053013566100866035|appliances.sewing...|janome|293.65|530496790|8e5f4f83-366c-4f7...|
|2019-11-01 00:00:01|      view|  17302664|2053013553853497655|                NULL| creed| 28.31|561587266|755422e7-9040-477...|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
only showing top 3 rows


In [0]:
from pyspark.sql.functions import count

df_n.groupBy("brand").agg(count("*").alias("Total_count")).show(3)


+------+-----------+
| brand|Total_count|
+------+-----------+
|  boss|        179|
| zotac|        998|
|shishi|         49|
+------+-----------+
only showing top 3 rows


In [0]:
filtered_df = df_n.filter(df_n.event_type == "purchase")

In [0]:
filtered_df.show(3)

+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|  brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+
|2019-11-01 00:00:41|  purchase|  13200605|2053013557192163841|furniture.bedroom...|   NULL| 566.3|559368633|d6034fa2-41fb-4ac...|
|2019-11-01 00:01:04|  purchase|   1005161|2053013555631882655|electronics.smart...| xiaomi|211.92|513351129|e6b7ce9b-1938-4e2...|
|2019-11-01 00:04:51|  purchase|   1004856|2053013555631882655|electronics.smart...|samsung|128.42|562958505|0f039697-fedc-40f...|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+
only showing top 3 rows


In [0]:
selected_df = filtered_df.select("user_id", "product_id", "price")
selected_df.show(5)

+---------+----------+------+
|  user_id|product_id| price|
+---------+----------+------+
|559368633|  13200605| 566.3|
|513351129|   1005161|211.92|
|562958505|   1004856|128.42|
|541854711|  26401669|109.66|
|557746614|   1801881| 488.8|
+---------+----------+------+
only showing top 5 rows


In [0]:
df_selected = df_n.select(
    "event_time",
    "event_type",
    "price",
    "brand"
)

In [0]:
df_filtered = df_selected.filter(
    (df_selected.event_type == "purchase") &
    (df_selected.price > 100)
)

In [0]:
df_filtered.show(3)

+-------------------+----------+------+-------+
|         event_time|event_type| price|  brand|
+-------------------+----------+------+-------+
|2019-11-01 00:00:41|  purchase| 566.3|   NULL|
|2019-11-01 00:01:04|  purchase|211.92| xiaomi|
|2019-11-01 00:04:51|  purchase|128.42|samsung|
+-------------------+----------+------+-------+
only showing top 3 rows


In [0]:
from pyspark.sql.functions import sum

brand_revenue = df_filtered.groupBy("brand") \
    .agg(sum("price").alias("total_revenue"))

In [0]:
brand_revenue.orderBy("total_revenue", ascending=False).show(10)


+-------+--------------------+
|  brand|       total_revenue|
+-------+--------------------+
|  apple|1.2748504469999993E8|
|samsung| 5.420126076999901E7|
| xiaomi|1.0258333020000026E7|
|   NULL|   9412464.630000005|
|     lg|   5195241.659999999|
| huawei|   4667545.509999995|
|   sony|   3775209.159999998|
|   oppo|   3488540.759999997|
|lucente|  3409932.9500000025|
|   acer|   3337005.520000002|
+-------+--------------------+
only showing top 10 rows


# Magic Commands

In [0]:
df_n.createOrReplaceTempView("ecommerce")


In [0]:
%sql
SELECT event_type, COUNT(*) AS count
FROM ecommerce
GROUP BY event_type
ORDER BY count DESC


event_type,count
view,63556110
cart,3028930
purchase,916939


### %fs – Explore files

In [0]:
%fs ls /Volumes/workspace/ecommerce/ecommerce_data/


path,name,size,modificationTime
dbfs:/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv,2019-Nov.csv,9006762395,1767896661000
dbfs:/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv,2019-Oct.csv,5668612855,1767896797000


### Export Results

In [0]:
brand_revenue.write.mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/brand_revenue_output.csv")


In [0]:
%fs ls /Volumes/workspace/ecommerce/ecommerce_data/

path,name,size,modificationTime
dbfs:/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv,2019-Nov.csv,9006762395,1767896661000
dbfs:/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv,2019-Oct.csv,5668612855,1767896797000
dbfs:/Volumes/workspace/ecommerce/ecommerce_data/brand_revenue_output.csv/,brand_revenue_output.csv/,0,1768066627429
